# Clustering Techniques — Simple Hands-On Notebook

This notebook is built to be followed step by step with almost no extra explanation needed.

**What we'll do, in order:**
1. Build a simple 2D "customer" dataset
2. Plot it BEFORE any clustering — just raw points
3. Use the Elbow Method to choose a good K
4. Run K-Means and plot the clusters it finds, with centroids
5. Run Hierarchical Clustering and draw a dendrogram
6. Cut the dendrogram and plot those clusters too

Run each cell top to bottom using **Shift + Enter**. Every line of code has a comment explaining exactly what it does.

## Step 1 — Import the Tools We Need

In [ ]:
# numpy helps us work with numbers and arrays
import numpy as np

# pandas lets us store data in a neat table, like a spreadsheet
import pandas as pd

# matplotlib is what we use to draw charts and pictures
import matplotlib.pyplot as plt

# make_blobs generates a simple practice dataset made of natural groups
from sklearn.datasets import make_blobs

# KMeans is the first clustering algorithm we're covering
from sklearn.cluster import KMeans

# AgglomerativeClustering is how scikit-learn runs Hierarchical Clustering
from sklearn.cluster import AgglomerativeClustering

# these two let us build and draw a dendrogram (the merge tree picture)
from scipy.cluster.hierarchy import linkage, dendrogram

# lock in the "randomness" so everyone gets the exact same numbers every time
np.random.seed(42)

# a simple message to confirm everything loaded properly
print("Ready to go!")

## Step 2 — Build a Simple Practice Dataset

We'll pretend we have data on 180 customers. Each customer has two numbers about them:

- `annual_spend` — how much they spend with us per year, in dollars
- `visits_per_month` — how often they visit per month

We don't tell the computer anything about groups — it only ever sees these two numbers per customer.

In [ ]:
# the 3 rough "centers" our fake customer groups will be built around
group_centers = [[-4, -2], [0, 3], [4, -3]]

# make_blobs generates points scattered around each center, like real customers
# would naturally cluster around a few typical spending habits
raw_points, _ = make_blobs(n_samples=180, centers=group_centers, cluster_std=0.8, random_state=42)

# turn the first raw column into a realistic-looking dollar amount
# np.clip stops any value from going below $50 (nobody spends negative money)
annual_spend = np.clip(raw_points[:, 0] * 150 + 950, 50, None).round(0)

# turn the second raw column into a realistic-looking visit count
# np.clip stops any value from going below 1 visit per month
visits_per_month = np.clip(raw_points[:, 1] * 1.3 + 12, 1, None).round(1)

# put both features into one table
customers = pd.DataFrame({
    "annual_spend": annual_spend,
    "visits_per_month": visits_per_month
})

# show the first 10 rows so we can see what it looks like
customers.head(10)

In [ ]:
# X holds the two features every clustering algorithm below will use
X = customers[["annual_spend", "visits_per_month"]].values

# print how many customers (rows) and features (columns) we have
print("Shape of X:", X.shape)

## Step 3 — Plot the Data BEFORE Any Clustering

In [ ]:
# make the picture a reasonable size
plt.figure(figsize=(7, 5))

# plot every customer as a single gray dot -- no groups, no colors yet
plt.scatter(X[:, 0], X[:, 1], color="gray", alpha=0.6)

# label both axes so the chart is easy to read
plt.xlabel("Annual Spend ($)")
plt.ylabel("Visits per Month")

# give the chart a title
plt.title("Raw Customer Data -- No Clustering Yet")

# actually display the picture
plt.show()

Even without being told anything, you can probably already see a few natural groupings just by eye. Clustering is how we make the computer find them too.

## Step 4 — Choose K With the Elbow Method

We'll try K = 1 through 8, record how "spread out" each clustering is (called inertia), and look for the bend in the curve.

In [ ]:
# the different K values we want to try
k_values = range(1, 9)

# we'll store each K's inertia score here
inertia_scores = []

# go through each K one at a time
for k in k_values:

    # create and train a KMeans model with this many clusters
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)

    # inertia_ is scikit-learn's built-in "how spread out are the clusters" score
    inertia_scores.append(model.inertia_)

# plot K on the x-axis against inertia on the y-axis
plt.figure(figsize=(7, 5))
plt.plot(list(k_values), inertia_scores, marker="o", color="#306998")

# label both axes
plt.xlabel("K (number of clusters)")
plt.ylabel("Inertia")

# give the chart a title
plt.title("The Elbow Method")

# actually display the picture
plt.show()

**Reading this chart:** look for the point where the line stops dropping steeply and starts to flatten out -- that bend is the "elbow." Here, it should be clearly around **K = 3**, which is exactly how many groups we built this data with.

## Step 5 — Run K-Means With K = 3, and Plot the Clusters

In [ ]:
# create a KMeans model with the K we chose from the elbow chart
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# train it on our customer data
kmeans.fit(X)

# labels_ tells us which cluster (0, 1, or 2) each customer landed in
cluster_labels = kmeans.labels_

# cluster_centers_ gives us the (x, y) position of each centroid
centroids = kmeans.cluster_centers_

# add the cluster label onto our table so we can see it next to the raw data
customers["cluster"] = cluster_labels

customers.head(10)

In [ ]:
# make the picture a reasonable size
plt.figure(figsize=(7, 5))

# plot every customer, colored by which cluster KMeans assigned them to
plt.scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap="viridis", alpha=0.7)

# plot the centroids as large red X marks so they stand out clearly
plt.scatter(centroids[:, 0], centroids[:, 1], c="red", marker="X", s=250, label="Centroids")

# label both axes
plt.xlabel("Annual Spend ($)")
plt.ylabel("Visits per Month")

# give the chart a title
plt.title("K-Means Clusters (K = 3)")

# show the legend so we know what the red X marks mean
plt.legend()

# actually display the picture
plt.show()

## Step 6 — Run Hierarchical Clustering, and Draw a Dendrogram

Now let's build the same kind of grouping a completely different way: by merging the closest points, step by step, from the bottom up.

In [ ]:
# linkage() does all the bottom-up merging, and records every merge that happened
# method="ward" is a standard choice -- it merges whichever pair keeps clusters as tight as possible
merge_history = linkage(X, method="ward")

# make the picture wide, since dendrograms need horizontal room
plt.figure(figsize=(11, 5))

# draw the dendrogram from our merge history
dendrogram(merge_history)

# label the axes
plt.xlabel("Individual Customers")
plt.ylabel("Merge Height (how different two groups are)")

# give the chart a title
plt.title("Dendrogram -- Every Possible Grouping, All at Once")

# actually display the picture
plt.show()

Reading tip: pick any height and imagine a horizontal cut line there -- the number of vertical lines it crosses is how many clusters you'd get. Notice there are 3 tall, obviously separate branches near the top, which is the dendrogram's own way of telling us K = 3 is a sensible choice here too.

## Step 7 — Cut the Tree Into 3 Clusters, and Plot Them

In [ ]:
# AgglomerativeClustering does the same bottom-up merging as our dendrogram,
# but stops automatically once exactly n_clusters groups are left
hierarchical = AgglomerativeClustering(n_clusters=3)

# fit_predict trains the model AND returns each customer's cluster label in one step
hierarchical_labels = hierarchical.fit_predict(X)

# make the picture a reasonable size
plt.figure(figsize=(7, 5))

# plot every customer, colored by which cluster Hierarchical Clustering assigned them to
plt.scatter(X[:, 0], X[:, 1], c=hierarchical_labels, cmap="viridis", alpha=0.7)

# label both axes
plt.xlabel("Annual Spend ($)")
plt.ylabel("Visits per Month")

# give the chart a title
plt.title("Hierarchical Clusters (cut at 3 groups)")

# actually display the picture
plt.show()

Compare this picture to the K-Means one in Step 5 -- on data with clearly separated groups like this, both algorithms usually agree closely, even though they got there in completely different ways.

## Recap

- **Clustering** has no labels to learn from -- it finds structure in the data by itself.
- **K-Means** needs you to choose K upfront. The **Elbow Method** (plotting inertia vs K) helps you pick a sensible one.
- **Hierarchical Clustering** doesn't need K upfront -- it builds a full merge tree (a dendrogram), and you decide afterward where to cut it.
- Both algorithms found the same 3 groups here, just by two very different routes.

That's the whole workflow: build data, plot it raw, choose K, cluster it two different ways, and compare.